# llm-training-kernels: GPU run (Phase 4a)

Runs the GPU-only parts: Triton FlashAttention tests (forward + new backward) and the attention benchmark.

1. **Runtime → Change runtime type → T4 GPU** (free tier is fine; an L4/A100 additionally runs the bf16 parts).
2. Run all cells. Cell 2 asks you to upload `llm-training-kernels.zip`.
3. The last cell downloads `gpu_results.zip`. Send that file back.

On a T4 (sm75) the bf16 tests and bf16 benchmarks are skipped automatically: T4 has no bf16 tensor cores.


In [ ]:
# 1. GPU + versions
!nvidia-smi
import platform

import torch
import triton

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
cap = torch.cuda.get_device_capability()
print(
    "torch",
    torch.__version__,
    "| CUDA",
    torch.version.cuda,
    "| triton",
    triton.__version__,
    "| python",
    platform.python_version(),
)
print("GPU:", torch.cuda.get_device_name(), "| compute capability:", cap)
print("bf16 parts will", "run" if cap >= (8, 0) else "be SKIPPED (needs sm80+, e.g. L4/A100)")

In [ ]:
# 2. Upload the repo zip and unpack it
import os
import shutil
import zipfile

from google.colab import files

os.chdir("/content")
uploaded = files.upload()  # choose llm-training-kernels.zip
name = next(iter(uploaded))
shutil.rmtree("/content/llm-training-kernels", ignore_errors=True)
zipfile.ZipFile(name).extractall("/content")
os.chdir("/content/llm-training-kernels")
print(sorted(os.listdir()))

In [ ]:
# 3. Install (keeps Colab's preinstalled torch/triton)
!pip install -q -e '.[dev]'
!python -c "import llmtk, llmtk.kernels.flash_attention_triton as t; print('llmtk ok; backward kernels:', hasattr(t, 'flash_backward_triton'))"

In [ ]:
# 4. Full test suite on GPU (log saved to benchmarks/results/pytest_gpu.txt)
!mkdir -p benchmarks/results
!python -m pytest -q -rxXs 2>&1 | tee benchmarks/results/pytest_gpu.txt | tail -60

In [ ]:
# 5. Attention benchmark: naive vs PyTorch-tiled vs Triton, fwd and fwd+bwd, latency + peak memory
runs = [("fp32", ""), ("fp32", "--causal")]
if torch.cuda.get_device_capability() >= (8, 0):
    runs += [("bf16", ""), ("bf16", "--causal")]
for dtype, flag in runs:
    name = f"attention_{dtype}{'_causal' if flag else ''}"
    print(f"===== {name}")
    !python -m benchmarks.bench_attention --dtype {dtype} {flag} --name {name}

In [ ]:
# 6. Download results
!cd benchmarks && zip -qr /content/gpu_results.zip results
files.download("/content/gpu_results.zip")